# 02 – Ön İşleme ve TF-IDF

01 numaralı not defterinde, referansla ortak kelime sayısının puanla ilişkili olduğunu ama tek başına yetmediğini gördük. Bu not defterinde kelime tabanlı yöntemi (TF-IDF) ayrıntılı tanıyacak ve şu soruya cevap arayacağız:

> **Metni temizlemek (ön işleme), kelime tabanlı puanlamayı iyileştirir mi?**

Adımlar:
1. Ön işleme nedir, hangi adımları deneyeceğiz?
2. TF-IDF nasıl çalışır? (Küçük bir örnekle)
3. Puanlama motorunun üç özelliği ve doğrusal regresyon
4. Başarıyı nasıl ölçüyoruz?
5. Beş farklı ön işleme seçeneğinin karşılaştırılması
6. Karar: Hangi ön işlemeyi kullanacağız?

### Nasıl çalıştırılır?
Bu dosyayı Drive'daki `CevapAnaliz` klasörüne (01 numaralı not defterinin yanına) yükle, çift tıklayıp Colab'da aç ve **Çalışma zamanı → Tümünü çalıştır** de. GPU gerekmez. Bütün not defteri yaklaşık 1 dakikada çalışır.

## 1. Kurulum

01 numaralı not defterindeki kurulumun aynısı: Drive'ı bağlar, kodları GitHub'dan çeker, Drive'daki veriyi kullanır. Ortak veri tablosu 01'de zaten oluşturulduğu için tekrar oluşturulmaz.

In [ ]:
import os, sys, subprocess
from pathlib import Path

COLAB = "google.colab" in sys.modules
if COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/CevapAnaliz")
    PROJE = Path("/content/CevapAnaliz")
    if PROJE.exists():
        subprocess.run(["git", "-C", str(PROJE), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/Semantik-Muhendisleri/CevapAnaliz.git", str(PROJE)], check=True)
    (DRIVE / "data").mkdir(parents=True, exist_ok=True)
    if not (PROJE / "data").exists():
        os.symlink(DRIVE / "data", PROJE / "data")
else:  # Kendi bilgisayarında çalıştırılırsa
    PROJE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DRIVE = PROJE

os.chdir(PROJE)
sys.path.insert(0, str(PROJE))
SEKIL = DRIVE / "sekiller"; SEKIL.mkdir(parents=True, exist_ok=True)
SONUC = DRIVE / "sonuclar"; SONUC.mkdir(parents=True, exist_ok=True)

# Ortak veri tablosu yoksa oluştur (01 numaralı not defteri çalıştırıldıysa zaten vardır)
if not (PROJE / "data" / "processed" / "ortak_veri.csv").exists():
    for betik in ["scripts/veri_indir.py", "scripts/ortak_bicim.py"]:
        sonuc = subprocess.run([sys.executable, betik], capture_output=True, text=True)
        print(sonuc.stdout[-1500:])
        if sonuc.returncode != 0:
            print(sonuc.stderr[-3000:])
            raise RuntimeError(f"{betik} başarısız oldu; hata mesajı yukarıda.")

import pandas as pd
df = pd.read_csv(PROJE / "data" / "processed" / "ortak_veri.csv", keep_default_na=False)
print(f"{len(df):,} cevap yüklendi.".replace(",", "."))

## 2. Ön işleme nedir?

Bilgisayar metni anlamaz; metni önce kelimelere böler, sonra bu kelimeleri sayılara çevirir. **Ön işleme**, metni bu işlemden önce temizleyip sadeleştirmektir. Amaç, aynı anlama gelen ama farklı yazılmış ifadelerin bilgisayara da aynı görünmesini sağlamaktır.

Deneyeceğimiz adımlar:

| Adım | Ne yapıyor? | Örnek |
|---|---|---|
| **Küçük harf ve noktalama temizliği** | Harfleri küçültür, noktalama işaretlerini siler. "doesn't" gibi kısaltmaları "does not" yapar | "Data members." → "data members" |
| **Gereksiz kelimeleri silme** (stop-word) | Çok sık geçen ama tek başına az anlam taşıyan kelimeleri atar | "the", "is", "of", "a" silinir |
| **Kelimeyi köküne indirme** (stemming) | Aynı kelimenin farklı hâllerini tek bir köke indirir | "functions", "function" → "function"; "simulates", "simulate" → "simul" |

Gereksiz kelime listesi olarak scikit-learn kütüphanesinin hazır İngilizce listesini (318 kelime), kök bulma için de en yaygın yöntem olan **Porter** algoritmasını kullanıyoruz.

Bir de dikkat edilmesi gereken nokta var: Hazır gereksiz kelime listesinde **"not", "no", "never"** gibi olumsuzluk kelimeleri ve **"more", "less", "before", "after"** gibi karşılaştırma kelimeleri de var. Bunlar silinince cümlenin anlamı değişebilir. Bu yüzden bir seçenekte bu kelimeleri **koruyarak** siliyoruz.

In [ ]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from nltk.stem import PorterStemmer

# Anlamı değiştirebileceği için silinmemesi gereken kelimeler
KORUNACAK = {"not", "no", "nor", "never", "none", "nothing", "cannot", "without", "neither", "nobody",
             "more", "less", "most", "few", "before", "after", "above", "below", "same", "other", "only", "all"}
GEREKSIZ = set(ENGLISH_STOP_WORDS)
GEREKSIZ_KORUMALI = GEREKSIZ - KORUNACAK

_kok_bulucu = PorterStemmer()
_kok_onbellek = {}
_KELIME = re.compile(r"[a-z0-9]+")

def kok(kelime):
    if kelime not in _kok_onbellek:
        _kok_onbellek[kelime] = _kok_bulucu.stem(kelime)
    return _kok_onbellek[kelime]

def on_isle(metin, gereksiz=None, kokle=False):
    """Metni küçük harfe çevirir, kelimelere böler; istenirse gereksiz kelimeleri siler ve kökleri bulur."""
    kelimeler = _KELIME.findall(metin.lower().replace("n't", " not"))
    if gereksiz:
        kelimeler = [k for k in kelimeler if k not in gereksiz]
    if kokle:
        kelimeler = [kok(k) for k in kelimeler]
    return " ".join(kelimeler)

SECENEKLER = {
    "Ham": {},
    "Stop-word silme": {"gereksiz": GEREKSIZ},
    "Stop-word silme\n(olumsuzluk korunur)": {"gereksiz": GEREKSIZ_KORUMALI},
    "Kök bulma": {"kokle": True},
    "Stop-word (korunarak)\n+ kök bulma": {"gereksiz": GEREKSIZ_KORUMALI, "kokle": True},
}

ornekler = [
    "The compiler doesn't create a default constructor if constructors are defined.",
    "A paperclip is harder than a penny, so it can scratch more minerals.",
]
for cumle in ornekler:
    print("Orijinal:", cumle)
    for ad, ayar in SECENEKLER.items():
        print(f"  {ad.replace(chr(10), ' '):<42} → {on_isle(cumle, **ayar)}")
    print()

**Dikkat:** İlk cümlede gereksiz kelimeler hazır listeyle silinince "doesn't create" ifadesindeki **"not" de siliniyor** ve cümle *"compiler does create default constructor"* hâline geliyor; yani **anlam tam tersine dönüyor**. İkinci cümlede de "than" ve "more" gidiyor ve karşılaştırma kayboluyor. Olumsuzluk ve karşılaştırma kelimelerini koruyan seçenek bu sorunu önlüyor.

Kök bulmada "simul", "defin" gibi gerçek olmayan kökler çıkabilir; bu bir sorun değil, önemli olan aynı kelimenin farklı hâllerinin aynı köke inmesidir.

## 3. TF-IDF nasıl çalışır?

TF-IDF, her kelimeye bir **önem puanı** verir. Bu puan iki şeyin çarpımıdır:

- **TF (terim sıklığı):** Kelime bu cevapta kaç kez geçiyor? Çok geçen kelime o cevap için önemlidir.
- **IDF (ters belge sıklığı):** Kelime *bütün cevaplarda* ne kadar yaygın? Herkesin kullandığı kelimeler ("the", "is") ayırt edici değildir, düşük puan alır. Az kullanılan kelimeler ("prototype", "simulates") ayırt edicidir, yüksek puan alır.

Böylece her cevap, kelimelerin önem puanlarından oluşan bir **sayı dizisine (vektöre)** dönüşür. İki cevabın benzerliği de bu iki dizinin **kosinüs benzerliği** ile ölçülür: 1'e yakınsa benzer, 0'a yakınsa farklı.

Küçük bir örnekle görelim: Bir referans cevap ve iki öğrenci cevabı.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

kucuk = {
    "referans": "A prototype simulates the behaviour of a software product.",
    "öğrenci 1": "A prototype program simulates parts of the desired software product.",
    "öğrenci 2": "It is used to test the software before it is finished.",
}
vektorlestirici = TfidfVectorizer()
matris = vektorlestirici.fit_transform(kucuk.values())

tablo = pd.DataFrame(matris.toarray().T, index=vektorlestirici.get_feature_names_out(), columns=kucuk.keys())
print("Her kelimenin her cevaptaki TF-IDF önem puanı (0 = o cevapta yok):")
display(tablo.round(2).sort_values("referans", ascending=False))

benzerlik = cosine_similarity(matris)
print(f"Referans – öğrenci 1 kosinüs benzerliği: {benzerlik[0, 1]:.2f}")
print(f"Referans – öğrenci 2 kosinüs benzerliği: {benzerlik[0, 2]:.2f}")

**Tabloyu nasıl okuyalım?**
- Üç cevapta da geçen **"the"** ve **"software"** düşük puan aldı: Herkes kullandığı için ayırt edici değiller.
- Sadece referansta geçen **"behaviour"** ve referansla öğrenci 1'de geçen **"prototype"**, **"simulates"** yüksek puan aldı.
- Öğrenci 1, referansın önemli kelimelerini kullandığı için benzerliği yüksek; öğrenci 2 neredeyse sadece "the" ve "software" ile örtüştüğü için benzerliği düşük.

Gerçek sistemde önem puanları bu üç cümleden değil, eğitim verisindeki **binlerce cevaptan** öğrenilir.

## 4. Puanlama motorunun üç özelliği ve doğrusal regresyon

2. hafta raporundaki FR-03 gereksinimine göre puan üç özellikten hesaplanıyor:

| Özellik | Ne ölçüyor? |
|---|---|
| **Kosinüs benzerliği** | TF-IDF vektörlerinin benzerliği (yukarıda gördüğümüz) |
| **N-gram örtüşmesi** | Referanstaki kelime ve ikili kelime gruplarının ne kadarı öğrenci cevabında geçiyor |
| **Uzunluk oranı** | Öğrenci cevabının referansa göre uzunluğu (en fazla 3 kat) |

Bu üç sayıyı tek bir puana çevirmek için **doğrusal regresyon** kullanıyoruz. En basit makine öğrenmesi modelidir:

> puan = a × kosinüs + b × n-gram + c × uzunluk + d

Model, eğitim verisindeki öğretmen puanlarına bakarak **a, b, c, d sayılarını kendisi öğrenir**: Öğretmen puanına en yakın sonucu veren ağırlıkları bulur. Üç özelliği bu şekilde tek modelde birleştirmeye **özellik birleştirme (feature fusion)** denir.

Mohler verisinde öğrenilen ağırlıklara bakalım:

In [ ]:
from sklearn.linear_model import LinearRegression
from src.puanlama.ozellikler import TfidfKosinus, ozellik_tablosu

OZELLIKLER = ["kosinus", "ngram", "uzunluk"]
mohler = df[df["veri_seti"] == "mohler"]
kosinus = TfidfKosinus().ogren(pd.concat([mohler["referans_cevap"], mohler["ogrenci_cevap"]]))
oz = ozellik_tablosu(mohler, kosinus)
model = LinearRegression().fit(oz[OZELLIKLER], mohler["puan"])

for ad, agirlik in zip(OZELLIKLER, model.coef_):
    print(f"{ad:<10} ağırlığı: {agirlik:+.3f}")
print(f"{'sabit':<10} (d)      : {model.intercept_:+.3f}")

**Ağırlıklar ne anlatıyor?** Artı işaretli bir ağırlık, o özellik arttıkça puanın da arttığını gösterir. Üç ağırlık da artı ve en büyük ağırlık kosinüs benzerliğinde: Model, puanı tahmin ederken en çok TF-IDF benzerliğine güveniyor. Referansla daha çok örtüşen cevap daha yüksek puan alıyor; bu beklediğimiz bir sonuç. Sabit sayının yüksek olması ise Mohler'deki puanların genel olarak yüksek olmasından (01'de gördüğümüz dengesizlik) kaynaklanıyor.

## 5. Başarıyı nasıl ölçüyoruz?

Bir modeli, eğitildiği verinin aynısıyla test etmek kopya çekmek gibidir: Cevapları ezberlemiş olabilir. Bu yüzden modeli **hiç görmediği cevaplarla** test ediyoruz:

- **Mohler:** Resmî bir eğitim/test ayrımı yok. Soruları 5 gruba ayırıyoruz; her seferinde 4 grupla eğitip kalan 1 grupla test ediyoruz ve bunu 5 kez tekrarlıyoruz (**5 katlı çapraz doğrulama**). Böylece model, test edildiği soruları eğitimde **hiç görmemiş** oluyor.
- **SciEntsBank ve Beetle:** Veri setinin kendi eğitim kümesiyle eğitip kendi test kümeleriyle ölçüyoruz:
  - **görülmemiş cevaplar** (`test_ua`): Eğitimde görülen sorulara verilmiş yeni cevaplar
  - **görülmemiş sorular** (`test_uq`): Eğitimde hiç görülmemiş sorular (daha zor)

Ölçütler (01'de gördüğümüz gibi, ortalama hata yanıltıcı olabileceği için):
- **Pearson korelasyonu:** Sistem puanı ile öğretmen puanı birlikte artıp azalıyor mu? (1 = mükemmel uyum)
- **QWK:** Puanlar birbiriyle ne kadar örtüşüyor? Büyük farkları küçük farklardan daha ağır cezalandırır. (1 = mükemmel uyum)

Hedefimiz (NFR-01): Pearson ≥ 0,70 ve QWK ≥ 0,70.

In [ ]:
import numpy as np
from sklearn.model_selection import GroupKFold
from src.puanlama.olcutler import olc

def egit_tahmin(egitim, test):
    kosinus = TfidfKosinus().ogren(pd.concat([egitim["referans_cevap"], egitim["ogrenci_cevap"]]))
    oz_egitim, oz_test = ozellik_tablosu(egitim, kosinus), ozellik_tablosu(test, kosinus)
    model = LinearRegression().fit(oz_egitim[OZELLIKLER], egitim["puan"])
    return np.clip(model.predict(oz_test[OZELLIKLER]), 0, 1)

def degerlendir(veri):
    satirlar = []
    mohler = veri[veri["veri_seti"] == "mohler"]
    tahmin = pd.Series(index=mohler.index, dtype=float)
    for egitim_i, test_i in GroupKFold(n_splits=5).split(mohler, groups=mohler["soru_id"]):
        tahmin.iloc[test_i] = egit_tahmin(mohler.iloc[egitim_i], mohler.iloc[test_i])
    satirlar.append({"test": "Mohler", **olc(mohler["puan"], tahmin, 5)})
    for veri_seti, ad in [("scientsbank", "SciEntsBank"), ("beetle", "Beetle")]:
        alt = veri[veri["veri_seti"] == veri_seti]
        egitim = alt[alt["bolum"] == "train"]
        for bolum, aciklama in [("test_ua", "görülmemiş cevaplar"), ("test_uq", "görülmemiş sorular")]:
            test = alt[alt["bolum"] == bolum]
            satirlar.append({"test": f"{ad} – {aciklama}", **olc(test["puan"], egit_tahmin(egitim, test), 2)})
    return satirlar

sonuclar = []
for ad, ayar in SECENEKLER.items():
    veri = df.copy()
    veri["referans_cevap"] = veri["referans_cevap"].map(lambda m: on_isle(m, **ayar))
    veri["ogrenci_cevap"] = veri["ogrenci_cevap"].map(lambda m: on_isle(m, **ayar))
    for satir in degerlendir(veri):
        sonuclar.append({"on_isleme": ad.replace("\n", " "), **satir})
    print("tamamlandı:", ad.replace("\n", " "))

sonuc = pd.DataFrame(sonuclar)
sonuc.round(4).to_csv(SONUC / "02_on_isleme_tfidf.csv", index=False, encoding="utf-8")
sira = [a.replace("\n", " ") for a in SECENEKLER]
for olcut in ["pearson", "qwk"]:
    print(f"\n{olcut.upper()}")
    display(sonuc.pivot(index="on_isleme", columns="test", values=olcut).loc[sira].round(3))

In [ ]:
import matplotlib.pyplot as plt

RENK = {"Mohler": "#2a78d6", "SciEntsBank – görülmemiş cevaplar": "#eb6834", "Beetle – görülmemiş cevaplar": "#1baf7a"}

fig, ax = plt.subplots(figsize=(13, 4.8))
genislik = 0.26
for i, (test, renk) in enumerate(RENK.items()):
    degerler = sonuc[sonuc["test"] == test].set_index("on_isleme").loc[sira, "pearson"]
    x = np.arange(len(sira)) + (i - 1) * genislik
    ax.bar(x, degerler, width=genislik - 0.03, color=renk, label=test)
ax.set_xticks(np.arange(len(sira)), list(SECENEKLER))
ax.set_ylabel("Pearson korelasyonu")
ax.set_ylim(0, 0.7)
for kenar in ["top", "right"]:
    ax.spines[kenar].set_visible(False)
ax.grid(axis="y", color="#e6e5e0"); ax.set_axisbelow(True)
ax.legend(frameon=False, loc="upper left", ncol=3)
ax.set_title("Ön işleme seçeneklerinin TF-IDF puanlamasına etkisi", loc="left", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(SEKIL / "02_on_isleme_tfidf.png", dpi=150, bbox_inches="tight")
plt.show()

**Ne görüyoruz?**

- **Gereksiz kelimeleri silmek her veri setinde sonucu kötüleştirdi.** Pearson değeri Mohler'de 0,51'den 0,45'e, SciEntsBank'te 0,42'den 0,38'e düştü. Sebebi yukarıda gördüğümüz örnek: "not", "more", "than" gibi küçük ama anlamlı kelimeler siliniyor. Kısa cevaplarda her kelime değerli.
- **Olumsuzluk ve karşılaştırma kelimeleri korunduğunda** testlerin çoğunda kaybın bir kısmı geri geliyor ama yine de hiç silmemekten daha kötü.
- **Kelimeyi köküne indirmek her veri setinde sonucu iyileştirdi.** Mohler'de 0,51 → 0,54, SciEntsBank'te (görülmemiş cevaplar) 0,42 → 0,46, görülmemiş sorularda 0,44 → 0,47. Çünkü "simulates" ile "simulate", "constructors" ile "constructor" artık aynı kelime sayılıyor.
- İkisini birlikte yapmak, yalnızca kök bulmaktan daha kötü: Gereksiz kelime silmenin zararı, kök bulmanın faydasını kısmen götürüyor.
- **Hiçbir seçenek hedefe (0,70) yaklaşamıyor.** En iyi seçenekle bile Pearson 0,34 (Beetle, görülmemiş sorular) ile 0,55 arasında kalıyor. Ön işleme küçük bir iyileştirme sağlıyor ama kelime tabanlı yöntemin sınırını aşamıyor.

## 6. Karar

| Karar | Gerekçe |
|---|---|
| **Kelime tabanlı yöntemde kök bulma kullanılacak** | Bütün veri setlerinde Pearson'ı 0,01–0,03 artırdı |
| **Gereksiz kelimeler silinmeyecek** | Bütün veri setlerinde sonucu kötüleştirdi; olumsuzluk kelimelerini silerek cümlenin anlamını tersine çevirebiliyor |
| **Anlam tabanlı modellere geçilecek** | En iyi ön işlemeyle bile Pearson 0,55'i geçmedi; hedef 0,70 |

Bir sonraki adımda kullanacağımız **Sentence-BERT** gibi yapay zekâ modellerinde ise ön işleme neredeyse hiç yapılmaz: Bu modeller doğal, bozulmamış cümlelerle eğitildiği için metni olduğu gibi (sadece fazla boşluklar temizlenmiş hâliyle) alır. Kök bulma ve kelime silme bu modellerin işini zorlaştırır.

## Özet

- **Ön işleme**, metni bilgisayarın daha iyi karşılaştırabilmesi için sadeleştirmektir. Beş seçenek denedik.
- **TF-IDF**, kelimelere "ne kadar ayırt edici?" sorusuna göre önem puanı verir; iki cevabın benzerliği kosinüs benzerliğiyle ölçülür.
- Puanlama motoru üç özelliği (kosinüs, n-gram, uzunluk) **doğrusal regresyonla birleştiriyor** (özellik birleştirme).
- **Kök bulma işe yarıyor, gereksiz kelime silme zarar veriyor.**
- Kelime tabanlı yöntemin en iyi sonucu Pearson 0,34–0,55; hedef 0,70'in oldukça altında.

Sonuçlar Drive'da `CevapAnaliz/sonuclar/02_on_isleme_tfidf.csv`, grafik `CevapAnaliz/sekiller/02_on_isleme_tfidf.png` dosyasına kaydedildi.

**Sonraki not defteri: 03 – Sentence-BERT.** İlk gerçek yapay zekâ modelimizi kullanacağız: Cümlelerin anlamını sayılara çeviren hazır bir model. 01'deki "ortak kelimesi yok ama 5/5 almış" cevapları bu modelin doğru tanıyıp tanımadığına bakacağız.